# Lab 04: Faces in the Clouds

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 1, *The Reality Generator*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-04.ipynb)

**What you will do:** time and count how often you spot a face that is not really there, compare your count with the book's benchmark, and watch an image classifier get nudged into confidently naming an object that was never in the picture.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

Step 1 (cloud watching) tells you to scan the sky "keeping your gaze away from the sun itself".
Never look directly at the sun, with or without this Lab.

## The experiment in the book

In Lab 04 you searched for faces in clouds, toast, camera sensor noise, and everyday objects
around the room, and noticed how automatic and hard to switch off the perception was once a face
appeared. The Lab explains this as pareidolia: a face detector tuned, for evolutionary reasons,
to favour false alarms over missed detections.

## Record your results

Step 3 (the noise field) and Step 4 (everyday objects) both give you something to log. Time
how long it takes for the first face-like shape to emerge in phone-camera sensor noise, and
count how many face-like objects you can find around you in two minutes.

In [ ]:
# example data: replace with your own timings and counts
pareidolia_log = pd.DataFrame({
    "task": ["noise field (Step 3)", "room search (Step 4)"],
    "seconds_to_first_face": [14, np.nan],   # example data (the room search isn't timed the same way)
    "faces_found_in_2min": [np.nan, 6],       # example data
})
display(pareidolia_log)

## Compare

Lab 04 reports: "Most people can find at least five face-like arrangements within a few
minutes" during the room search (Step 4).

In [ ]:
found = pareidolia_log.loc[pareidolia_log.task == "room search (Step 4)", "faces_found_in_2min"].iloc[0]
print(f"You found {found:.0f} face-like objects.")
if found >= 5:
    print("At or above the book's 'most people find at least five' benchmark.")
else:
    print("Below the book's typical benchmark -- try again over a longer search, or in a room")
    print("with more electrical fittings, patterned fabric, or machinery (common sources of")
    print("accidental two-eyes-a-nose-a-mouth arrangements).")

## The AI side

Your face detector is tuned to fire even on random noise -- a "better safe than sorry" setting,
the book explains, because missing a real face is costlier than imagining one. Deep image
classifiers show a related failure: Nguyen, Yosinski, and Clune (2015) found that networks
trained on real photographs can be driven to report near-certain confidence for images that
are meaningless to a human eye. Below, we start from pure random pixels and nudge them, one
small gradient step at a time, purely to raise a small pretrained classifier's confidence in
one fixed, arbitrary category -- with no real object of that kind ever appearing.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import torch
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights).eval()
categories = weights.meta["categories"]

# Standard ImageNet normalisation statistics, used by this and most torchvision classifiers
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

torch.manual_seed(2026)
target_idx = 130   # a fixed, arbitrary ImageNet class index -- whatever it turns out to be, it never appears in random pixels
image = torch.rand(1, 3, 224, 224, requires_grad=True)
before_image = image.detach().clone()

optimizer = torch.optim.Adam([image], lr=0.05)
for step in range(150):
    optimizer.zero_grad()
    probs = model((image - mean) / std).softmax(dim=1)
    loss = -probs[0, target_idx]
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        image.clamp_(0, 1)

with torch.no_grad():
    before_p = model((before_image - mean) / std).softmax(dim=1)[0, target_idx].item()
    after_p = model((image - mean) / std).softmax(dim=1)[0, target_idx].item()

label = categories[target_idx]
fig, axes = plt.subplots(1, 2, figsize=(6, 3.2))
axes[0].imshow(before_image[0].permute(1, 2, 0).numpy())
axes[0].set_title(f"Random start\n'{label}': {before_p:.1%}", fontsize=9)
axes[1].imshow(image.detach()[0].permute(1, 2, 0).numpy())
axes[1].set_title(f"After 150 optimiser steps\n'{label}': {after_p:.1%}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Confidence that this image is '{label}' rose from {before_p:.1%} to {after_p:.1%},")
print("even though no real object was ever in the image -- only nudged random pixels.")

The optimiser reached this result by the most literal route possible: adjust pixels wherever
the gradient says confidence will rise. Nguyen, Yosinski, and Clune (2015) showed the same
result at scale, across many categories. The disanalogy matters: your face-pareidolia bias is
a single, ancient, evolutionarily tuned setting for one specific category (faces), calibrated
by the cost of missing a predator or a rival. This classifier has no comparable stake in the
matter -- it was never taught that "no object present" is even a valid answer, so gradient
ascent can walk it to false certainty about almost anything, not just faces.

## Pool the class data

One room search gives a noisy count. Pool the class: each person spends two minutes counting
face-like objects in their own room (an anonymous ID, not a name, plus the count), downloads
the results as CSV and loads it here.

In [ ]:
import io
csv_text = """id,faces_found_2min
P01,7
P02,4
P03,9
P04,6
P05,5
"""  # example data: replace with your class CSV, e.g. pd.read_csv("pareidolia_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
boot = [df.faces_found_2min.sample(len(df), replace=True).mean() for _ in range(5000)]
lo_ci, hi_ci = np.percentile(boot, [2.5, 97.5])
print(f"Class mean: {df.faces_found_2min.mean():.1f} faces "
      f"(95% bootstrap interval {lo_ci:.1f}-{hi_ci:.1f}, n = {len(df)})")
df.faces_found_2min.plot.hist(bins=6, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.axvline(5, color="firebrick", linestyle="--", label="book: 'most people find at least five'")
plt.xlabel("face-like objects found in 2 minutes"); plt.legend(); plt.show()

## Questions to think about

1. In the gradient-ascent demo, the optimiser needed no real object of the target category to reach high confidence. Contrast this with your own face-pareidolia. Is there ever "nothing at all" driving your perception, or does your brain always need at least some ambiguous shape (a knot in wood, a pattern of shadows) to work with?
2. The book frames face detection as a signal-detection trade-off between sensitivity and specificity. Using your own room-search count, would turning your "is this a face?" threshold down (stricter, fewer false alarms) have found more or fewer of the objects you counted -- and would you have missed any?
3. Why might the same gradient-ascent technique used above be far less effective at fooling a human observer, even though both systems (your brain and the classifier) can be pushed toward confident false detections?
4. Oliva and Torralba (2007) showed that scene context guides object recognition (you expect a desk in an office). Pick one face you found in Step 4 and describe the everyday function of that object that first drew your attention to it, before you consciously noticed the eyes-nose-mouth arrangement.

## Challenge

Google's DeepDream project amplified whatever a network already half-detected, turning faint
textures into hallucinated eyes and faces throughout an image. Recreate the everyday version:
photograph a highly textured surface (tree bark, a stucco wall, a marbled countertop) and spend
two full minutes searching it for every face-like arrangement you can find, marking each one.

In [ ]:
# Example data: replace with your own count and a short description of each face you marked.
my_texture_search = pd.DataFrame({
    "surface": ["tree bark"],                                    # example data
    "faces_marked": [4],                                          # example data
    "most_convincing": ["a knot plus two smaller knots above it"],  # example data
})
display(my_texture_search)
print("Highly textured, high-contrast surfaces tend to yield more candidate 'faces' than smooth")
print("ones, because they offer more chance arrangements of dark and light patches for the face")
print("detector's low threshold to latch onto.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-04.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")